In [ ]:
import os
os.environ["ANTHROPIC_API_KEY"]  # set in .env


In [ ]:
import os
import json
import base64
import anthropic

# =====================================================
# CONFIG
# =====================================================

API_KEY = os.environ["ANTHROPIC_API_KEY"]

MODEL = "claude-haiku-4-5"

FILE_PATH = r"C:\Users\hieu.nguyengia\Downloads\365\data\Bonne Nuit Hotel Contract 2026.pdf"


MAX_TOKENS = 16000

TEMPERATURE = 0

# =====================================================
# MIME TYPE DETECTION
# =====================================================

def get_media_type(file_path):

    ext = os.path.splitext(file_path)[1].lower()

    media_types = {
        ".pdf": "application/pdf",
        ".txt": "text/plain",
        ".docx": "application/vnd.openxmlformats-officedocument.wordprocessingml.document"
    }

    if ext not in media_types:
        raise ValueError(f"Unsupported file type: {ext}")

    return media_types[ext]

# =====================================================
# READ FILE
# =====================================================

def load_file_as_base64(file_path):

    with open(file_path, "rb") as f:
        file_data = f.read()

    return base64.b64encode(file_data).decode("utf-8")

# =====================================================
# PROMPT
# =====================================================

PROMPT = """
You are an expert hospitality FIT contract extraction engine.

Your task is to extract ONLY FIT (Free Independent Traveler)
hotel contract information from the uploaded document.

Return STRICT VALID JSON ONLY.

==================================================
CORE EXTRACTION RULES
==================================================

1. ONLY extract information explicitly written in the document.

2. DO NOT hallucinate, infer, estimate, normalize, or invent values.

3. IGNORE all NON-FIT information:
- GIT
- GROUP
- SERIES
- CORPORATE
- WHOLESALE
- MICE
- EVENT
- ALLOTMENT
- CREW
- CONTRACTOR
- OTA CAMPAIGN

4. If a table contains both FIT and NON-FIT data:
extract ONLY FIT rows and columns.

5. USE ONLY the EXACT schema keys defined below.

6. DO NOT:
- create additional keys
- rename keys
- restructure arrays
- restructure objects
- generate nested attributes not defined in schema

7. NEVER generate undefined keys such as:
- age_bracket
- breakfast_fee
- room_rate_fit
- policy_item
- surcharge_data
- hotel_package
- package_detail
- extra_information

8. Preserve EXACTLY:
- room names
- currencies
- VND formatting
- percentages
- date ranges
- rate formatting
- punctuation
- multilingual wording

9. If information is missing:
- use empty string ""
- use empty array []
- NEVER guess

10. NEVER explain.

11. NEVER summarize.

12. NEVER return markdown.

13. NEVER wrap JSON inside code blocks.

14. Schema consistency is CRITICAL across all hotels.

15. Return STRICT VALID JSON ONLY.

==================================================
SEMANTIC RULES
==================================================

A. facilities
= permanent hotel facilities/amenities.

Examples:
- swimming pool
- gym
- spa
- sauna
- wifi
- parking
- beach access
- kids club

B. included_benefits
= benefits included with booking/rate/package.

Examples:
- breakfast included
- airport transfer
- welcome drink
- complimentary dinner
- spa voucher
- late checkout

DO NOT duplicate facilities into included_benefits.

C. meal_information
ONLY use for:
- restaurant information
- gala dinner
- compulsory dinner
- buffet pricing
- special meal pricing

DO NOT duplicate breakfast inclusion here
if breakfast already exists inside room seasonal_rates.

Breakfast inclusion inside room seasonal_rates
is the PRIMARY SOURCE OF TRUTH.

==================================================
OUTPUT STRUCTURE
==================================================

{
  "hotel_information": {
    "hotel_name": "",
    "brand_group": "",
    "address": "",
    "district": "",
    "city": "",
    "country": "",
    "website": "",
    "email": [],
    "phone_numbers": [],
    "tax_code": "",
    "representatives": [
      {
        "name": "",
        "position": ""
      }
    ]
  },

  "contract_information": {
    "contract_name": "",
    "contract_number": "",
    "validity": {
      "start_date": "",
      "end_date": ""
    },
    "applicable_market": [],
    "confidentiality_clause": "",
    "internet_rate_restriction": ""
  },

  "room_types": [
    {
      "room_name": "",
      "room_code": "",
      "room_category": "",
      "description": "",
      "size_sqm": "",
      "number_of_rooms": "",
      "bed_types": [],
      "view": "",
      "window_type": "",
      "max_occupancy": "",
      "standard_occupancy": "",
      "room_features": [],

      "seasonal_rates": [
        {
          "season_name": "",
          "date_range": "",
          "occupancy_type": "",
          "market_type": "FIT",
          "rate_plan": "",
          "currency": "",
          "price": "",
          "meal_included": [],
          "tax_included": "",
          "service_charge_included": "",
          "rate_notes": []
        }
      ]
    }
  ],

  "extra_bed_policy": {
    "available": "",
    "max_extra_beds_per_room": "",

    "extra_bed_rates": [
      {
        "season": "",
        "currency": "",
        "price": "",
        "conditions": []
      }
    ]
  },

  "child_policy": {

    "infant_policy": [
      {
        "age_range": "",
        "currency": "",
        "price": "",
        "conditions": []
      }
    ],

    "child_free_policy": [
      {
        "age_range": "",
        "currency": "",
        "price": "",
        "conditions": []
      }
    ],

    "child_breakfast_policy": [
      {
        "age_range": "",
        "currency": "",
        "price": "",
        "conditions": []
      }
    ],

    "child_extra_bed_policy": [
      {
        "age_range": "",
        "currency": "",
        "price": "",
        "conditions": []
      }
    ],

    "adult_definition": "",
    "child_definition": "",

    "baby_cot_policy": [
      {
        "currency": "",
        "price": "",
        "conditions": []
      }
    ]
  },

  "meal_information": {

    "restaurant_information": [],

    "meal_rates": [
      {
        "meal_type": "",
        "currency": "",
        "price": "",
        "conditions": []
      }
    ]
  },

  "included_benefits": [
    {
      "benefit_type": "",
      "details": "",
      "applicable_room_types": [],
      "conditions": []
    }
  ],

  "facilities": [
    {
      "facility_name": "",
      "availability": "",
      "notes": ""
    }
  ],

  "transportation_services": [
    {
      "service_type": "",
      "vehicle_type": "",
      "route": "",
      "currency": "",
      "price": "",
      "conditions": []
    }
  ],

  "checkin_checkout_policy": {

    "checkin_time": "",
    "checkout_time": "",

    "early_checkin_policy": [
      {
        "time_range": "",
        "currency": "",
        "price": "",
        "conditions": []
      }
    ],

    "late_checkout_policy": [
      {
        "time_range": "",
        "currency": "",
        "price": "",
        "conditions": []
      }
    ]
  },

  "cancellation_policy": [
    {
      "period": "",
      "currency": "",
      "price": "",
      "percentage": "",
      "conditions": []
    }
  ],

  "payment_policy": {

    "payment_terms": [
      {
        "details": ""
      }
    ],

    "deposit_policy": [
      {
        "details": ""
      }
    ],

    "refund_policy": [
      {
        "details": ""
      }
    ]
  },

  "peak_period_surcharge": [
    {
      "occasion": "",
      "date_range": "",
      "currency": "",
      "price": "",
      "percentage": "",
      "conditions": []
    }
  ],

  "restrictions": [
    {
      "type": "",
      "details": ""
    }
  ],

  "important_notes": [],

  "raw_season_definitions": [
    {
      "season_name": "",
      "date_ranges": []
    }
  ],

  "special_hotel_feature": {
    "feature_name": "",
    "feature_category": "",
    "details": ""
  },

  "metadata": {
    "document_language": [],
    "document_type": "",
    "extraction_confidence": "",
    "contains_fit_rates": "",
    "contains_child_policy": "",
    "contains_meal_policy": "",
    "contains_transport_service": "",
    "contains_peak_surcharge": "",
    "contains_special_feature": ""
  }
}

==================================================
ROOM EXTRACTION RULES
==================================================

Extract ALL FIT room types.

Extract when available:
- room size
- room quantity
- occupancy
- bed type
- room view
- balcony
- terrace
- internal window
- suite
- villa
- family room
- connecting room
- private pool
- bathtub
- kitchen

==================================================
FIT RATE EXTRACTION RULES
==================================================

Extract ONLY FIT rates.

Preserve:
- seasonality
- date ranges
- occupancy
- breakfast inclusion
- taxes
- service charge
- nett/gross wording
- promotional notes
- minimum stay

==================================================
CHILD POLICY RULES
==================================================

Use ONLY:
- age_range
- currency
- price
- conditions

NEVER create additional fields.

==================================================
FACILITY EXTRACTION RULES
==================================================

Facilities are permanent hotel amenities.

Extract:
- wifi
- swimming pool
- gym
- spa
- sauna
- tea/coffee
- minibar
- safe box
- shuttle
- parking
- laundry
- kids club
- beach access

==================================================
SPECIAL HOTEL FEATURE RULES
==================================================

ONLY ONE special feature is allowed.

Use:
"special_hotel_feature"

This field is ONLY for the MOST UNIQUE characteristic
that strongly differentiates the hotel.

Examples:
- onsen experience
- yoga retreat
- herbal bath
- glamping
- floating breakfast
- coffee farm experience
- eco lodge
- meditation retreat
- cooking class

If no special feature exists:
return empty strings.

==================================================
OUTPUT CLEANING RULES
==================================================

- Normalize whitespace.
- Preserve multilingual wording.
- Preserve numeric formatting exactly.
- Preserve currency formatting exactly.
- Return STRICT VALID JSON ONLY.
"""

# =====================================================
# CLAUDE CLIENT
# =====================================================

client = anthropic.Anthropic(
    api_key=API_KEY
)

# =====================================================
# LOAD FILE
# =====================================================

filename = os.path.basename(FILE_PATH)

media_type = get_media_type(FILE_PATH)

base64_file = load_file_as_base64(FILE_PATH)

# =====================================================
# API CALL
# =====================================================

print("Sending document to Claude...")

response = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    messages=[
        {
            "role": "user",
            "content": [

                {
                    "type": "document",
                    "source": {
                        "type": "base64",
                        "media_type": media_type,
                        "data": base64_file
                    }
                },

                {
                    "type": "text",
                    "text": PROMPT
                }

            ]
        }
    ]
)

# =====================================================
# RAW RESPONSE
# =====================================================

result_text = response.content[0].text

# =====================================================
# SAVE RAW RESPONSE
# =====================================================
base_name = os.path.splitext(filename)[0] 

raw_output_file = f"{base_name}_raw_response.txt"

with open("raw_response.txt", "w", encoding="utf-8") as f:
    f.write(result_text)

print("Raw response saved -> raw_response.txt")

# =====================================================
# JSON PARSE
# =====================================================

try:

    parsed_json = json.loads(result_text)

    output_file = os.path.splitext(filename)[0] + ".json"

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(parsed_json, f, ensure_ascii=False, indent=2)

    print(f"SUCCESS -> {output_file}")

except Exception as e:

    print("\nJSON PARSE ERROR")
    print(e)

    print("\nClaude returned invalid JSON.")
    print("Check raw_response.txt")